# A2 — Knowledge-Base Demo (fill this)
Show OCR quality on a sample and one working retrieval example.

In [ ]:

from pathlib import Path
import sys
import os

REPO_DIR = Path.cwd().parent

SRC_DIR = REPO_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

os.chdir(REPO_DIR)

from doc_agent import config, pipeline
from doc_agent.retrieval.retriever import Retriever

print(f"Repository: {REPO_DIR}")
print(f"Python: {sys.executable}")
print(f"Working directory: {Path.cwd()}")

In [ ]:
cfg = config.load()

print("Configuration loaded.")
print("OCR backend:", cfg["ocr"].get("backend"))
print("Embedding model:", cfg["embed"].get("model"))
print("Index type:", cfg["index"].get("type"))

In [ ]:
print("Building knowledge base...")

pipeline.build_knowledge_base(cfg)

print("Knowledge-base build completed.")

In [ ]:
raw_path = (
    REPO_DIR
    / "data"
    / "raw"
    / "higher_math_page_0016.png"
)

preprocessed_path = (
    REPO_DIR
    / "data"
    / "interim"
    / "preprocessed"
    / "higher_math"
    / "higher_math_page_0016.png"
)

layout_path = (
    REPO_DIR
    / "reports"
    / "figures"
    / "layout"
    / "higher_math"
    / "higher_math_page_0016.png"
)

for path in [raw_path, preprocessed_path, layout_path]:
    print(path)
    print("exists:", path.is_file())
    print()

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
raw_image = Image.open(raw_path)

print("Original page size:", raw_image.size)

plt.figure(figsize=(10, 14))
plt.imshow(raw_image)
plt.axis("off")
plt.title("1. Original scanned page — page_0014")
plt.show()

In [ ]:
preprocessed_image = Image.open(preprocessed_path)

print("Preprocessed page size:", preprocessed_image.size)

plt.figure(figsize=(10, 14))
plt.imshow(preprocessed_image)
plt.axis("off")
plt.title("2. Preprocessed page — page_0016")
plt.show()

In [ ]:
layout_image = Image.open(layout_path)

print("Layout visualization size:", layout_image.size)

plt.figure(figsize=(10, 14))
plt.imshow(layout_image)
plt.axis("off")
plt.title("3. Layout detection — page_0016")
plt.show()

In [ ]:
from doc_agent.contracts import Page
from doc_agent.vision import layout

page = Page(
    id="higher_math__higher_math_page_0016",
    image_path=str(preprocessed_path),
    doc_id="higher_math",
)

regions = layout.detect([page], cfg)

print(f"Detected {len(regions)} regions on page_0005")
for i, region in enumerate(regions[:10]):
    print(i, region.page_id, region.kind, region.bbox)


In [ ]:
from doc_agent.vision import ocr

reader = ocr.Reader(cfg)
ocr_results = []

for i, region in enumerate(regions):
    if region.kind not in {"text", "heading"}:
        continue

    text = reader.transcribe_region(region)
    if text:
        ocr_results.append(
            {
                "region_index": i,
                "kind": region.kind,
                "bbox": region.bbox,
                "text": text,
            }
        )

print(f"OCR extracted text from {len(ocr_results)} regions.")
for item in ocr_results[:5]:
    print(f"\n[{item['region_index']}] {item['kind']} {item['bbox']}")
    print(item['text'][:400])
    print("-" * 80)


In [ ]:
from doc_agent import pipeline

# Run the earlier stages again only if your pipeline exposes
# intermediate OCR output. Otherwise, inspect the generated
# preprocessed/OCR artifacts according to your project structure.

print("OCR backend:", cfg["ocr"].get("backend"))
print("OCR model/language:", cfg["ocr"].get("languages", cfg["ocr"].get("model")))

In [ ]:
from doc_agent.vision import ocr
from doc_agent import contracts

print("OCR implementation loaded:")
print(ocr.Reader)

In [ ]:
retriever = Retriever(cfg)

query = "সেট এবং ফাংশন সম্পর্কিত ধারণা"

results = retriever.retrieve(query, k=3)

print(f"Query: {query}")
print(f"Retrieved {len(results)} results\n")

for rank, hit in enumerate(results, start=1):
    page = hit.page_ids[0] if hit.page_ids else "unknown"

    print(f"#{rank}")
    print(f"Score: {hit.score:.3f}")
    print(f"Page:  {page}")
    print(f"ID:    {hit.id}")
    print("-" * 70)
    print(hit.text)
    print("=" * 70)

In [ ]:
query = "বহুপদী কাকে বলে? বহুপদীর উদাহরণ"

results = retriever.retrieve(query, k=3)

for rank, hit in enumerate(results, start=1):
    page = hit.page_ids[0] if hit.page_ids else "unknown"

    print(f"\n#{rank} | score={hit.score:.3f} | page={page}")
    print(hit.text)
    print("-" * 80)